<a href="https://colab.research.google.com/github/rymarinelli/bayesbench-notebook/blob/master/bayesbench_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BayesBench: Bayesian Early Stopping for a Recommender A/B Test

**BayesBench** is a Python package designed to make A/B testing and evaluation faster and more efficient using Bayesian sequential testing. Traditional A/B tests require you to pre-compute a fixed sample size and wait for all the data to arrive before making a decision. **BayesBench** replaces this with a sequential approach: it continuously monitors the incoming stream of results (like user clicks, LLM judge scores, or agent success rates) and updates a probability distribution over which model is better.

By constantly asking "are we confident enough to stop yet?", BayesBench can often declare a winner in a fraction of the time and cost of a fixed-horizon test. It provides lightweight utilities to track these Bayesian posteriors, calculate credible intervals, and trigger early stopping safely. This is incredibly useful for recommender systems, LLM evaluation, and testing autonomous agents where compute or API costs are high.

**The scenario:** you run a movie streaming service. Production currently serves a **popularity-based recommender** ("show everyone the most-rated movies"). Your team built a **candidate recommender** — a trained matrix factorization model, personalized per user — and wants to roll it out. Before flipping 100% of traffic to it, you run an online A/B test.

A fixed-horizon A/B test would commit to evaluating, say, 400 user sessions no matter what. **Bayesian sequential testing** instead re-checks the evidence after every session: it maintains a posterior belief over "P(candidate beats production)" and stops the test the moment that belief is confident enough — cutting the number of users exposed to the (possibly worse, or already-proven) losing model.

Critically, this isn't a one-shot offline comparison of two frozen models. **Both models keep learning from live traffic as the test runs** — a real recommender pipeline doesn't stop retraining just because an A/B test is in progress. In this notebook we:

1. Load real interaction data ([MovieLens ratings](https://huggingface.co/datasets/ashraq/movielens_ratings)) and split users into an **offline training pool** (fits both models initially, like a nightly batch job) and a **live-traffic pool** (simulates incoming user sessions for the A/B test).
2. Fit the **production model** (global popularity ranking) and the **candidate model** (implicit-feedback ALS matrix factorization — the same family of algorithm behind Spotify's early recommender, and the foundation of the `implicit` Python library) on the training pool.
3. Score each live session with **Hit Rate@10**: did the model's top-10 list contain the movie this user went on to rate highly? A clean 0/1 outcome per session.
4. **Fold each session's outcome back into the training data and periodically retrain both models** — a continuous-training loop, so later sessions are scored by models that have seen more recent data than the ones scoring the first session.
5. Run a Beta-Bernoulli Bayesian sequential test over the stream of sessions, and stop as soon as the winner is decisive.

Everything runs on CPU in well under a minute — no GPU, no API keys, no model downloads.

In [ ]:
# Uncomment if running in a fresh environment (e.g. Colab):
# %pip install -q datasets numpy pandas matplotlib

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datasets import load_dataset

# These are the parameters for the tutorial. Please feel free to tweek them to make it a bit more interative.
SEED = 42
N_MOVIES = 300            # keep the item catalog small enough to train quickly
MIN_RATINGS = 8           # users need enough history to build a profile + hold one out
K = 10                    # recommend the top-10 movies
N_TRAIN_USERS = 500       # simulate a "cold" initial model
N_EVAL_USERS = 1500       # simulate a longer live-traffic stream
FAVORITE_THRESHOLD = 4.0  # a rating >= this counts as "the user loved it" (also the implicit-ALS preference signal)
N_FACTORS = 10            # latent dimensions learned by the matrix factorization model
ALS_ITERS = 12            # alternating least-squares sweeps (user step + item step = 1 iter)
MF_REG = 1.0              # L2 regularization on the latent factors
ALPHA = 3.0               # confidence boost for observed (rated) items vs. unobserved ones
RETRAIN_EVERY = 25        # fold in new sessions and refit both models every N sessions
CONFIDENCE = 0.90         # stop as soon as P(candidate > production) or vice versa reaches this
CREDIBILITY = 0.95        # width of the reported credible interval

rng = np.random.default_rng(SEED)


## 1. Load ratings and split into offline / live-traffic pools

We restrict to the 300 most-rated movies (keeps training fast) and to users with at least `MIN_RATINGS` ratings among them (enough history for a profile, plus one to hold out as the "session outcome"). Users are then split: most go into the **training pool** that fits the models offline; a held-out slice becomes the **live-traffic pool** the A/B test evaluates against — mirroring a real deploy where you train on historical logs and test on new, unseen sessions.

In [ ]:
ds = load_dataset("ashraq/movielens_ratings", split="train")
ratings = ds.to_pandas()[["user_id", "movie_id", "rating", "title"]].drop_duplicates(["user_id", "movie_id"])
print(f"Total ratings: {len(ratings):,}")

top_movies = ratings["movie_id"].value_counts().head(N_MOVIES).index
ratings = ratings[ratings["movie_id"].isin(top_movies)]
movie_ids = sorted(top_movies.tolist())
movie_idx = {m: i for i, m in enumerate(movie_ids)}
movie_title = ratings.drop_duplicates("movie_id").set_index("movie_id")["title"].to_dict()

user_counts = ratings.groupby("user_id").size()
eligible_users = user_counts[user_counts >= MIN_RATINGS].index.to_numpy(copy=True)
rng.shuffle(eligible_users)

train_users = eligible_users[:N_TRAIN_USERS]
eval_users = eligible_users[N_TRAIN_USERS:N_TRAIN_USERS + N_EVAL_USERS]
print(f"Movies in catalog: {len(movie_ids)}")
print(f"Offline training pool: {len(train_users):,} users")
print(f"Live-traffic pool: {len(eval_users):,} users")

## 2. Fit the two models (initial offline training)

- **Production — Popularity:** rank movies by how many pool users rated them. Same list for everyone; no personalization.
- **Candidate — Implicit-feedback ALS matrix factorization:**  
It learns a low-dimensional latent vector for every user and every movie such that `user_vector · movie_vector` approximates whether that user will like that movie, fit by alternating least squares (fix the item vectors, solve for the optimal user vectors in closed form; fix the user vectors, solve for the optimal item vectors; repeat). This is the same algorithm family behind `implicit` (the standard Python ALS library) and Spotify's early recommender.

  We treat a rating `>= FAVORITE_THRESHOLD` as a positive preference signal (1) and anything else as 0, and give **every observed rating** — liked or not — higher confidence than an unrated (unobserved) movie, via the `ALPHA` weight. This is the standard Hu–Koren–Volinsky "implicit feedback" formulation: it optimizes directly for *which items a user prefers*, which matches our Hit Rate@K evaluation much better than fitting raw star ratings would.

We wrap this in a `fit_models()` function because we'll call it again later, every time fresh live-traffic data is folded back in.

In [ ]:
def als_solve(target_n, other_factors, idx_lists, pref_lists, reg, n_factors, alpha):
    """One ALS half-step: solve a closed-form ridge regression per row, given the other side's factors fixed."""
    YtY = other_factors.T @ other_factors
    out = np.zeros((target_n, n_factors))
    I = np.eye(n_factors) * reg
    for k in range(target_n):
        idxs = idx_lists[k]
        if not idxs:
            continue
        prefs = np.array(pref_lists[k])
        F = other_factors[idxs]
        A = YtY + F.T @ (F * alpha) + I          # implicit-ALS confidence-weighted normal equations
        b = F.T @ (prefs * (1 + alpha))
        out[k] = np.linalg.solve(A, b)
    return out


def fit_models(ratings_pool):
    """Fit the candidate (implicit-ALS matrix factorization) and production (popularity) models."""
    users = ratings_pool["user_id"].unique()
    user_idx = {u: i for i, u in enumerate(users)}
    n_users = len(users)

    user_items = [[] for _ in range(n_users)]
    user_pref = [[] for _ in range(n_users)]
    item_users = [[] for _ in range(len(movie_ids))]
    item_pref = [[] for _ in range(len(movie_ids))]
    for row in ratings_pool.itertuples(index=False):
        u, i = user_idx[row.user_id], movie_idx[row.movie_id]
        pref = 1.0 if row.rating >= FAVORITE_THRESHOLD else 0.0
        user_items[u].append(i); user_pref[u].append(pref)
        item_users[i].append(u); item_pref[i].append(pref)

    fit_rng = np.random.default_rng(SEED)
    item_factors = fit_rng.normal(0, 0.1, size=(len(movie_ids), N_FACTORS))
    user_factors = fit_rng.normal(0, 0.1, size=(n_users, N_FACTORS))
    for _ in range(ALS_ITERS):
        user_factors = als_solve(n_users, item_factors, user_items, user_pref, MF_REG, N_FACTORS, ALPHA)
        item_factors = als_solve(len(movie_ids), user_factors, item_users, item_pref, MF_REG, N_FACTORS, ALPHA)

    popularity_rank = [
        movie_idx[m] for m in ratings_pool.groupby("movie_id").size().sort_values(ascending=False).index
    ]
    return item_factors, popularity_rank


# The pool of "known" ratings the models are trained on. It starts as the offline
# training pool; run_ab_test() (Section 5) grows its own copy as live sessions arrive.
initial_known_ratings = ratings[ratings["user_id"].isin(train_users)].copy()
initial_item_factors, initial_popularity_rank = fit_models(initial_known_ratings)

print(f"Item latent-factor matrix: {initial_item_factors.shape}  ({N_FACTORS} factors per movie)")
print(f"Fitted on {initial_known_ratings['user_id'].nunique():,} users")
print("Most popular movie in the training pool:", movie_title[movie_ids[initial_popularity_rank[0]]])

## 3. Score a live session — Hit Rate@10

For each live-traffic user we simulate a session: hide one movie they rated `>= FAVORITE_THRESHOLD` (the "outcome" of the session — what they actually wanted), show the model everything else in their history, and ask it to recommend `K` movies. A hit means the recommender surfaced the movie the user actually loved.

In [ ]:
def simulate_session(user_id, eval_ratings, item_factors, popularity_rank):
    user_ratings = eval_ratings[eval_ratings["user_id"] == user_id]
    favorites = user_ratings[user_ratings["rating"] >= FAVORITE_THRESHOLD]
    if favorites.empty:
        return None

    target = favorites.sample(1, random_state=int(user_id) % (2**32 - 1)).iloc[0]
    target_idx = movie_idx[target["movie_id"]]

    profile = user_ratings[user_ratings["movie_id"] != target["movie_id"]]
    profile_idx = [movie_idx[m] for m in profile["movie_id"]]
    profile_pref = [1.0 if r >= FAVORITE_THRESHOLD else 0.0 for r in profile["rating"]]
    if not profile_idx:
        return None

    # Candidate: this live-traffic user was never in the training pool, so we "fold them in" --
    # solve the same closed-form ALS ridge regression for a single new user, holding the fitted
    # item factors fixed. This is exactly how production MF systems score a user between full
    # retrains: no need to refit the whole model just to serve one more session.
    user_vec = als_solve(
        target_n=1, other_factors=item_factors,
        idx_lists=[profile_idx], pref_lists=[profile_pref],
        reg=MF_REG, n_factors=N_FACTORS, alpha=ALPHA,
    )[0]

    mf_scores = item_factors @ user_vec
    mf_scores[profile_idx] = -np.inf  # never re-recommend something already seen
    mf_topk = set(np.argsort(-mf_scores)[:K])
    mf_hit = 1.0 if target_idx in mf_topk else 0.0

    # Production: same popularity list for everyone, minus what they've already seen
    pop_topk = []
    for idx in popularity_rank:
        if idx in profile_idx:
            continue
        pop_topk.append(idx)
        if len(pop_topk) == K:
            break
    pop_hit = 1.0 if target_idx in set(pop_topk) else 0.0

    return mf_hit, pop_hit


MODEL_A_NAME, MODEL_B_NAME = "Candidate (implicit-ALS matrix factorization)", "Production (popularity)"

## 4. Bayesian sequential test

Each session's hit/miss is a Bernoulli trial. We keep a `Beta(alpha, beta)` posterior over each model's true hit rate, starting from an uninformative `Beta(1, 1)` prior, and update it after every session. `P(candidate > production)` is Monte Carlo-estimated by sampling from both posteriors; we stop the test the moment that probability (or its complement) crosses `CONFIDENCE`.

In [ ]:
def bayesian_sequential_test(scores_a, scores_b, confidence, credibility, mc_samples=10_000, seed=0):
    rng = np.random.default_rng(seed)
    alpha_a = beta_a = 1.0
    alpha_b = beta_b = 1.0
    trace = []
    winner = None
    stopped_at = len(scores_a)

    for i, (sa, sb) in enumerate(zip(scores_a, scores_b)):
        alpha_a += sa; beta_a += 1 - sa
        alpha_b += sb; beta_b += 1 - sb

        samp_a = rng.beta(alpha_a, beta_a, mc_samples)
        samp_b = rng.beta(alpha_b, beta_b, mc_samples)
        p_a_beats_b = float((samp_a > samp_b).mean())

        trace.append({
            "session": i + 1,
            "p_a_beats_b": p_a_beats_b,
            "mean_a": alpha_a / (alpha_a + beta_a),
            "mean_b": alpha_b / (alpha_b + beta_b),
        })

        if winner is None:
            if p_a_beats_b >= confidence:
                winner, stopped_at = "A", i + 1
                break
            if p_a_beats_b <= 1 - confidence:
                winner, stopped_at = "B", i + 1
                break

    samp_a = rng.beta(alpha_a, beta_a, mc_samples)
    samp_b = rng.beta(alpha_b, beta_b, mc_samples)
    tail = (1 - credibility) / 2 * 100

    result = {
        "winner": winner,
        "p_a_beats_b": float((samp_a > samp_b).mean()),
        "mean_a": alpha_a / (alpha_a + beta_a),
        "mean_b": alpha_b / (alpha_b + beta_b),
        "ci_a": (float(np.percentile(samp_a, tail)), float(np.percentile(samp_a, 100 - tail))),
        "ci_b": (float(np.percentile(samp_b, tail)), float(np.percentile(samp_b, 100 - tail))),
        "sessions_used": stopped_at,
        "sessions_planned": len(scores_a),
        "efficiency": 1 - stopped_at / len(scores_a) if len(scores_a) else 0.0,
    }
    return result, trace

## 5. Run the A/B test — with continuous retraining

We replay the live-traffic pool in a fixed (shuffled) order, as if sessions arrived one at a time. `run_ab_test()` does this for a given retraining cadence:

1. Score both models with their **current** parameters (`item_factors`, `popularity_rank`) — the candidate scores a session by folding the user in (Section 3), the same closed-form solve used during training.
2. Once the outcome is known (the user did or didn't engage with the recommendation), fold that user's full rating history into the pool — this is new training signal, exactly like a production event log growing in real time.
3. Every `retrain_every` sessions, call `fit_models()` again on the updated pool and swap in the refreshed models for the next batch of sessions.

Passing a `retrain_every` larger than the whole live-traffic pool means step 3 never fires — the models stay frozen at their initial fit. We'll use that in Section 7 to check whether the retraining is actually earning its keep. We still collect the full stream first so the notebook always has data to plot, then run the sequential test over it; that's equivalent to actually stopping (and no longer retraining) the moment `sessions_used` is reached.

In [ ]:
eval_ratings = ratings[ratings["user_id"].isin(eval_users)]


def run_ab_test(retrain_every):
    pool = initial_known_ratings.copy()
    item_factors, pop_rank = initial_item_factors, initial_popularity_rank

    scores_a, scores_b, retrain_points = [], [], []
    for user_id in eval_users:
        outcome = simulate_session(user_id, eval_ratings, item_factors, pop_rank)
        if outcome is None:
            continue
        scores_a.append(outcome[0])
        scores_b.append(outcome[1])

        # The session's outcome is now known — fold it into the training pool.
        pool = pd.concat([pool, eval_ratings[eval_ratings["user_id"] == user_id]], ignore_index=True)

        if len(scores_a) % retrain_every == 0:
            item_factors, pop_rank = fit_models(pool)
            retrain_points.append(len(scores_a))

    result, trace = bayesian_sequential_test(scores_a, scores_b, CONFIDENCE, CREDIBILITY)
    return {
        "scores_a": scores_a, "scores_b": scores_b,
        "retrain_points": retrain_points, "final_pool_users": pool["user_id"].nunique(),
        "result": result, "trace": trace,
    }


main_run = run_ab_test(RETRAIN_EVERY)
scores_a, scores_b = main_run["scores_a"], main_run["scores_b"]
retrain_points = main_run["retrain_points"]
result, trace = main_run["result"], main_run["trace"]

print(f"Simulated sessions: {len(scores_a)}")
print(f"Retrains performed: {len(retrain_points)}  (after sessions {retrain_points})")
print(f"Training pool grew from {N_TRAIN_USERS:,} to {main_run['final_pool_users']:,} users")
print(f"{MODEL_A_NAME} hit rate@{K}: {np.mean(scores_a):.3f}")
print(f"{MODEL_B_NAME} hit rate@{K}: {np.mean(scores_b):.3f}")

result

## 6. Results — would you ship the candidate?

In [ ]:
winner_name = MODEL_A_NAME if result["winner"] == "A" else MODEL_B_NAME if result["winner"] == "B" else "Inconclusive"
print(f"Winner: {winner_name}")
print(f"P({MODEL_A_NAME} > {MODEL_B_NAME}) = {result['p_a_beats_b']:.3f}")
print(f"Sessions used: {result['sessions_used']}/{result['sessions_planned']}  "
      f"({result['efficiency']:.1%} of the planned A/B test avoided)")
retrains_before_stop = sum(1 for rp in retrain_points if rp < result["sessions_used"])
print(f"Model retrains before the test stopped: {retrains_before_stop}")
print(f"{MODEL_A_NAME}: hit rate={result['mean_a']:.3f}  {int(CREDIBILITY*100)}% CI={result['ci_a']}")
print(f"{MODEL_B_NAME}: hit rate={result['mean_b']:.3f}  {int(CREDIBILITY*100)}% CI={result['ci_b']}")

if result["winner"] == "A":
    print("\n--> Ship the candidate: stop the test, promote it to 100% of traffic.")
elif result["winner"] == "B":
    print("\n--> Keep production: the candidate underperforms, don't roll it out.")
else:
    print("\n--> Inconclusive at this sample size: keep collecting sessions, or lower CONFIDENCE.")

In [ ]:
sessions = [t["session"] for t in trace]
p_curve = [t["p_a_beats_b"] for t in trace]
mean_a_curve = [t["mean_a"] for t in trace]
mean_b_curve = [t["mean_b"] for t in trace]

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

ax = axes[0]
ax.plot(sessions, p_curve, color="#4F46E5", marker="o", markersize=3, label="P(candidate > production)")
ax.axhline(CONFIDENCE, color="#059669", linestyle="--", label=f"Ship candidate ≥ {CONFIDENCE:.0%}")
ax.axhline(1 - CONFIDENCE, color="#DC2626", linestyle="--", label=f"Keep production ≥ {1 - CONFIDENCE:.0%}")
ax.axhline(0.5, color="#94A3B8", linestyle=":", label="Even odds")
for j, rp in enumerate(retrain_points):
    if rp <= max(sessions):
        ax.axvline(rp, color="#F59E0B", linestyle=":", linewidth=1, alpha=0.7,
                    label="Model retrained" if j == 0 else None)
if result["winner"] is not None:
    ax.axvline(result["sessions_used"], color="#1E293B", linestyle="-", linewidth=1, alpha=0.6)
    ax.annotate("test stopped\nhere", xy=(result["sessions_used"], 0.05), fontsize=8, ha="left")
ax.set_ylim(0, 1)
ax.set_xlabel("Live sessions evaluated")
ax.set_ylabel("Posterior probability")
ax.set_title(f"P({MODEL_A_NAME} > {MODEL_B_NAME})")
ax.legend(fontsize=7, loc="center right")

ax = axes[1]
ax.plot(sessions, mean_a_curve, color="#4F46E5", label=MODEL_A_NAME)
ax.plot(sessions, mean_b_curve, color="#EF553B", label=MODEL_B_NAME)
for j, rp in enumerate(retrain_points):
    if rp <= max(sessions):
        ax.axvline(rp, color="#F59E0B", linestyle=":", linewidth=1, alpha=0.7,
                    label="Model retrained" if j == 0 else None)
ax.set_ylim(0, 1)
ax.set_xlabel("Live sessions evaluated")
ax.set_ylabel(f"Posterior mean hit rate@{K}")
ax.set_title("Posterior mean convergence")
ax.legend(fontsize=8)

fig.suptitle("BayesBench: Bayesian early stopping for a recommender A/B test", fontweight="bold")
fig.tight_layout()
plt.show()

## 7. Does retraining actually help?

Section 5 asserted that continuous retraining makes this closer to a real production system, but didn't measure whether it changes the outcome. Let's check directly: replay the **identical** live-traffic stream (same users, same shuffle, same held-out "favorite" per user — nothing here is re-randomized) under two policies:

- **Static** — `retrain_every` set larger than the whole live-traffic pool, so both models stay frozen at their initial fit for the entire test.
- **Continuously retrained** — the `RETRAIN_EVERY`-session cadence used above.

Any difference in the numbers below is attributable to the extra ~0–350 sessions' worth of data each model got to train on, not to noise in which sessions were sampled.

In [ ]:
static_run = run_ab_test(retrain_every=10**9)  # never fires within N_EVAL_USERS sessions

def _summarize(run, label):
    r = run["result"]
    winner = MODEL_A_NAME if r["winner"] == "A" else MODEL_B_NAME if r["winner"] == "B" else "Inconclusive"
    return {
        "policy": label,
        f"{MODEL_A_NAME} hit rate": np.mean(run["scores_a"]),
        f"{MODEL_B_NAME} hit rate": np.mean(run["scores_b"]),
        "winner": winner,
        "sessions_used": r["sessions_used"],
        "sessions_planned": r["sessions_planned"],
    }

comparison = pd.DataFrame([
    _summarize(static_run, "Static (no retraining)"),
    _summarize(main_run, f"Continuously retrained (every {RETRAIN_EVERY})"),
]).set_index("policy")
comparison

In [ ]:
cand_delta = np.mean(main_run["scores_a"]) - np.mean(static_run["scores_a"])
prod_delta = np.mean(main_run["scores_b"]) - np.mean(static_run["scores_b"])
speed_delta = static_run["result"]["sessions_used"] - main_run["result"]["sessions_used"]

print(f"Candidate hit-rate change from retraining: {cand_delta:+.3f}")
print(f"Production hit-rate change from retraining: {prod_delta:+.3f}")
print(f"Sessions-to-decision change from retraining: {speed_delta:+d} "
      f"({'faster' if speed_delta > 0 else 'slower' if speed_delta < 0 else 'no change'})")

if abs(cand_delta) < 0.02 and abs(speed_delta) <= 5:
    print(
        "\nAt this data scale (~50 sessions between retrains, on a mature 12,000-user "
        "training pool) retraining makes little measurable difference here -- the pool "
        "barely moves as a fraction of what the models were already fit on. That's a "
        "real, useful finding: it tells you retraining cadence isn't the lever to pull "
        "in this regime. The mechanism still matters in settings with faster drift or a "
        "smaller/colder initial training pool, where each new batch of sessions is a much "
        "larger fraction of what the model knows."
    )
else:
    print(
        "\nRetraining measurably changed the outcome here -- evidence that, even at this "
        "scale, keeping the models fed with live data is doing real work."
    )

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Calculate cumulative hit rates for both models (retrained vs static)
sessions_range = np.arange(1, len(main_run["scores_a"]) + 1)

cum_hits_main_a = np.cumsum(main_run["scores_a"]) / sessions_range
cum_hits_static_a = np.cumsum(static_run["scores_a"]) / sessions_range

cum_hits_main_b = np.cumsum(main_run["scores_b"]) / sessions_range
cum_hits_static_b = np.cumsum(static_run["scores_b"]) / sessions_range

plt.figure(figsize=(12, 6))

# Plot Candidate model performance
plt.plot(cum_hits_main_a, label="Candidate (Retrained)", color="#4F46E5", linewidth=2)
plt.plot(cum_hits_static_a, label="Candidate (Static)", color="#4F46E5", linestyle="--", alpha=0.6)

# Plot Production model performance
plt.plot(cum_hits_main_b, label="Production (Retrained)", color="#EF553B", linewidth=2)
plt.plot(cum_hits_static_b, label="Production (Static)", color="#EF553B", linestyle="--", alpha=0.6)

# Mark retrain points
for i, rp in enumerate(retrain_points):
    plt.axvline(rp, color="#F59E0B", linestyle=":", alpha=0.8,
                label="Model Retrained" if i == 0 else "")

plt.title("Cumulative Hit Rate@10: Continuous Retraining vs. Static")
plt.xlabel("Live Sessions Evaluated")
plt.ylabel("Cumulative Hit Rate")
plt.legend(loc="best")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Why this matters for MLOps

A fixed-horizon A/B test here would commit to 400 sessions regardless of how lopsided the result is. The Bayesian sequential test instead stops as soon as the posterior is confident — in a typical run that's a large majority of the planned traffic **never exposed to the losing model**, and a decision available in a fraction of the wall-clock time. For a real rollout that translates directly into less user-facing risk and a faster ship/no-ship call.

Layering continuous retraining on top makes the setup closer to a real production system: the models being compared aren't static artifacts frozen at deploy time, they're being updated by the same pipeline that's usually running against live event logs. Section 7 measured whether that actually changes anything here, rather than just asserting it — read those numbers, not this paragraph, for the honest answer at this data scale. The mechanism itself generalizes to systems with faster drift (trending content, seasonal demand, a cold-start-heavy user base) or a smaller/newer training pool, where each batch of live sessions is a much larger share of what the model knows — that's where a sequential test's ability to react to a model that's improving (or degrading) mid-test earns its keep over a fixed-horizon test decided in advance.

While tuning the candidate model, some hyperparameter settings produced an early confident call (after just 2–8 sessions) that pointed the *opposite* direction from what the full 395-session sample actually showed — a real, known failure mode of naive sequential testing, where a short unlucky streak crosses the confidence bar before the noise has had a chance to average out. The `SEED`/hyperparameters shipped here don't exhibit that, but a production system would typically guard against it with a minimum-sample floor before any stopping decision is allowed to fire (see Next steps below).

## Next steps

- Tighten or loosen `CONFIDENCE` to trade decision speed against certainty.
- Add a minimum-sample floor to `bayesian_sequential_test` (e.g. don't allow a stop before ~30 sessions per arm) to guard against the early-noise failure mode described above.
- Shrink `RETRAIN_EVERY` further, or shrink `N_TRAIN_USERS` so each retrain is a bigger relative jolt to the model — both make the effect measured in Section 7 easier to see.
- Sweep `N_FACTORS`, `ALPHA`, or `MF_REG` and watch the hit-rate gap (and the stopping point) move — and notice how easily a badly-tuned candidate can lose to the popularity baseline entirely.
- Replace either model with your own — the sequential test only needs a `[0, 1]`-valued score per session, so it works unchanged for click-through, conversion, or any other online-metric A/B test.
- For the full interactive experience (multi-task suites, model ranking, agent/tool-use benchmarks, custom HF datasets), see the [Streamlit demo](https://github.com/rymarinelli/bayesbench-demo).
- The underlying package is [`bayesbench`](https://github.com/rymarinelli/bayesbench), which this notebook's `bayesian_sequential_test` mirrors in miniature.

## Part 2: Evaluating Agents with BayesBench

For **Agents**, evaluation is even more costly. A single evaluation might involve spinning up a sandbox, letting the agent execute bash commands, browse the web, and then checking the final environment state (e.g.,SWE-bench or WebArena).

Here, early stopping is critical. If Agent A has a new reasoning loop that drastically improves its success rate over the baseline Agent B, we want to know as soon as statistically possible to avoid wasting hours of compute time.

### Understanding the Environment: MountainCar-v0

In the **MountainCar-v0** environment, an underpowered car must reach a flag at the top of a steep hill. The engine isn't strong enough to drive straight up, so the only way to succeed is to rock back and forth to build up **momentum**.

- **Agent B (Random Baseline)** takes random actions. It almost never builds enough coordinated momentum to reach the top within the 200-step limit.
- **Agent A (Heuristic Momentum)** uses a simple rule: it checks the car's current velocity and always pushes in that same direction. This effectively builds momentum and solves the environment very consistently.

Because the performance gap between these two agents is so massive, the Bayesian test recognizes Agent A's superiority almost instantly. Let's visualize how the probability curve crosses the 99% confidence threshold in just 4 evaluation tasks.

In [ ]:
import random
import numpy as np

try:
    import gymnasium as gym
except ImportError:
    import gym

# 1. Define Agents for MountainCar
class MountainCarHeuristicAgent:
    def __init__(self, name, noise_level=0.1):
        self.name = name
        self.noise_level = noise_level

    def act(self, obs):
        # obs is [position, velocity]
        # Heuristic: always push in the direction of current velocity to build momentum
        if random.random() < self.noise_level:
            return random.choice([0, 1, 2])
        return 2 if obs[1] > 0 else 0

class MountainCarRandomAgent:
    def __init__(self, name):
        self.name = name

    def act(self, obs):
        return random.choice([0, 1, 2])

# 2. Setup the Environment and Evaluation Task queue
agent_a = MountainCarHeuristicAgent("Agent A (Heuristic Momentum)", noise_level=0.1)
agent_b = MountainCarRandomAgent("Agent B (Random Baseline)")

n_gym_tasks = 200
agent_a_scores = []
agent_b_scores = []

env = gym.make("MountainCar-v0")

# 3. Evaluate the agents in MountainCar
def evaluate_agent(agent, env):
    obs = env.reset()
    if isinstance(obs, tuple):
        obs = obs[0]

    steps = 0
    done = False
    while not done and steps < 200:
        action = agent.act(obs)
        result = env.step(action)

        # Handle both old (4-tuple) and new (5-tuple) gym APIs
        if len(result) == 5:
            obs, reward, terminated, truncated, _ = result
            done = terminated or truncated
        else:
            obs, reward, done, _ = result
            terminated = done

        steps += 1

        # MountainCar success is reaching the flag (terminated=True) before 200 steps
        if terminated and steps < 200:
            return 1

    return 0 # Failed to reach the top in time

random.seed(42)
np.random.seed(42)

for _ in range(n_gym_tasks):
    agent_a_scores.append(evaluate_agent(agent_a, env))
    agent_b_scores.append(evaluate_agent(agent_b, env))

env.close()

# Suppose each sandbox evaluation takes 2 minutes and costs $0.10 in API calls.
time_per_task_mins = 2
cost_per_task_dollars = 0.10

# 4. Run the Bayesian Sequential Test
agent_result, agent_trace = bayesian_sequential_test(
    agent_a_scores, agent_b_scores, confidence=0.99, credibility=0.95
)

tasks_run = agent_result['sessions_used']
saved_tasks = n_gym_tasks - tasks_run

print(f"Agent Eval Winner: {agent_result['winner']}")
print(f"Gym Tasks Evaluated: {tasks_run}/{n_gym_tasks}")
print(f"Efficiency (Tasks Avoided): {agent_result['efficiency']:.1%}")
print(f"--> Compute Hours Saved: {(saved_tasks * time_per_task_mins) / 60:.1f} hours")
print(f"--> Evaluation Budget Saved: ${saved_tasks * cost_per_task_dollars:.2f}")


In [ ]:
import matplotlib.pyplot as plt

# Extract traces from the agent evaluation
sessions = [t["session"] for t in agent_trace]
p_curve = [t["p_a_beats_b"] for t in agent_trace]
mean_a_curve = [t["mean_a"] for t in agent_trace]
mean_b_curve = [t["mean_b"] for t in agent_trace]

# We used 99% confidence for the agent test
agent_confidence = 0.99

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# Left plot: Probability that Agent A is better than Agent B
ax = axes[0]
ax.plot(sessions, p_curve, color="#4F46E5", marker="o", markersize=5, label="P(Agent A > Agent B)")
ax.axhline(agent_confidence, color="#059669", linestyle="--", label=f"Stop (A wins) ≥ {agent_confidence:.0%}")
ax.axhline(1 - agent_confidence, color="#DC2626", linestyle="--", label=f"Stop (B wins) ≤ {1 - agent_confidence:.0%}")
ax.axhline(0.5, color="#94A3B8", linestyle=":", label="Even odds")

if agent_result["winner"] is not None:
    ax.axvline(agent_result["sessions_used"], color="#1E293B", linestyle="-", linewidth=1, alpha=0.6)
    ax.annotate("test stopped\nhere", xy=(agent_result["sessions_used"], 0.05), fontsize=9, ha="center")

ax.set_ylim(0, 1.05)
ax.set_xlabel("Gym tasks evaluated")
ax.set_ylabel("Posterior probability")
ax.set_title("P(Heuristic Agent > Random Agent)")
ax.legend(fontsize=8, loc="center right")

# Right plot: Posterior mean success rate of each agent
ax = axes[1]
ax.plot(sessions, mean_a_curve, color="#4F46E5", marker="o", markersize=4, label="Agent A (Heuristic)")
ax.plot(sessions, mean_b_curve, color="#EF553B", marker="o", markersize=4, label="Agent B (Random Baseline)")

if agent_result["winner"] is not None:
    ax.axvline(agent_result["sessions_used"], color="#1E293B", linestyle="-", linewidth=1, alpha=0.6)

ax.set_ylim(0, 1.05)
ax.set_xlabel("Gym tasks evaluated")
ax.set_ylabel("Posterior mean success rate")
ax.set_title("Agent Skill Convergence")
ax.legend(fontsize=8, loc="upper left")

fig.suptitle("BayesBench: Bayesian Early Stopping for Autonomous Agents", fontweight="bold")
fig.tight_layout()
plt.show()